In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects
from pathlib import Path
import HumSpectra.fluorescence as fl
import HumSpectra.ultraviolet as uv
import HumSpectra.utilits as ut
import HumSpectra.math_stat as st

In [ ]:
date_list = ["2024","2025"]
fluo_settings_list = [0,1]
iqr_setings_list = [2,2.5]
data_list = []

In [ ]:
def plot_heat_map(data: pd.DataFrame,
                  ax= None,
                  xlabel: bool = True,
                  ylabel: bool = True,
                  title: bool = True,
                  name: str|None = None):
    """
    :param data: DataFrame, спектр флуоресценции
    :param q: float, квантиль, определяющий экстремальные выбросы
    :return: ax: Axes, ось графика matplotlib.pyplot
    Функция маскирует экстремальные выбросы нулями, и рисует 2D тепловой график флуоресценции
    """
    
    data_copy = data.copy()
    
    if name is None:
        
        if "name" in data_copy.attrs:
            name = data_copy.attrs["name"]
        else:
            name = "fluo_spectra"
    
    EM_wavelengths = data_copy.index.to_numpy(dtype="int")
    EX_wavelengths = data_copy.columns.to_numpy(dtype="int")

    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(5.7, 4.8))
    ax.pcolormesh(EM_wavelengths, EX_wavelengths, data_copy.T, shading="gouraud",
                  cmap=plt.get_cmap('rainbow'))
    if xlabel:
        ax.set_xlabel("λ испускания, нм")
    if ylabel:
        ax.set_ylabel("λ возбуждения, нм")
    if title:
        ax.set_title(f"{name}")

    return ax

In [ ]:
fluo_spectra_2Dlist = []
uv_spectra_2Dlist = []

for i,date in enumerate(date_list):
    wdir_date = Path.cwd().parent / "Initial data" / date

    fluo_path = Path.joinpath(wdir_date, "Fluorescence")
    uv_path = Path.joinpath(wdir_date, "UV","Individual spectra")

    fluo_spectra = ut.load_spectra_data(fluo_path, fl.read_fluo_3d, index_col=fluo_settings_list[i])
    uv_spectra = ut.load_spectra_data(uv_path, uv.read_csv_uv, spectra_type="absorption")
    
    fluo_spectra_cut = [fl.cut_spectra(fl_spectra,255,450,255,600) 
                        for fl_spectra in fluo_spectra]
    
    fluo_spectra_2Dlist.append(fluo_spectra_cut)
    uv_spectra_2Dlist.append(uv_spectra)

In [ ]:
for fl_list in fluo_spectra_2Dlist:
    fl.plot_optic_spectra_by_subclass(fl_list,fl.plot_heat_map)

In [ ]:
plot_heat_map(fl.cut_spectra
              (fluo_spectra_2Dlist[0][0]
               ,255,450,255,600)
            )